# Phase 2 — The Model
## Brain Tumour MRI Classification
====================================================================

Define the network. Nothing is trained here and no data decision is made — the
dataset was fixed in Phase 1 and is not touched.

This is a separate phase because the architecture is the part of this project
that was *derived* rather than downloaded, and that derivation is the claim the
report rests on.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch

from src import config, viz
from src.config import CLASSES, DEVICE, DROPOUT, IMG_SIZE
from src.manifest import read as read_manifest
from src.model import (BrainTumourNet, ConvBlock, count_parameters,
                       receptive_field)

M = read_manifest()
if M is None:
    raise RuntimeError('no run manifest on disk -- run Phase 1 first')
print(f"run {M['hash']}   dataset {M['dataset']}   "
      f"{M['n_images']} images / {M['n_patients']} patients / {len(M['classes'])} classes")
print(f"split      {M['split_sizes']}")
print(f"patients   {M['patients_per_split']}")

run b9dcd147c12e   dataset figshare-cheng-2017   3064 images / 233 patients / 3 classes
split      {'train': 2099, 'val': 352, 'test': 613}
patients   {'train': 159, 'val': 27, 'test': 47}


In [ ]:
# 1. WHAT WAS DERIVED, AND WHAT IS ONLY IMPORTED
"""
The one property worth restating is negative: there is no pretrained backbone
anywhere in this model. Every weight starts from Kaiming initialisation and is
learned from the 2,099 training images Phase 1 left behind. A fine-tuned ResNet
would score several points higher and would demonstrate nothing about whether
the layers were understood, which is the actual claim being made.

"""
model = BrainTumourNet(num_classes=len(CLASSES), dropout=DROPOUT, deep=True)
print(model)

BrainTumourNet(
  (features): BrainTumourCNN(
    (block1): ConvBlock(
      (conv): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): ReLU(inplace=True)
    )
    (block2): ConvBlock(
      (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): ReLU(inplace=True)
    )
    (block3): ConvBlock(
      (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): ReLU(inplace=True)
    )
    (block4): ConvBlock(
      (conv): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)


In [ ]:
# 2. WHERE THE PARAMETERS ACTUALLY ARE
"""
The reason is the global average pool. Flattening a 16x16x256 feature map into
the head would produce 65,536 inputs and a first dense layer of eight million
weights. Averaging each channel to a single number instead produces 256 inputs
and a first layer of 32,768. The head becomes a rounding error, and the capacity
sits where it should -- in the convolutional stages that actually look at the
image.
"""
enc = sum(p.numel() for p in model.features.parameters())
head = sum(p.numel() for p in model.head.parameters())
total, trainable = count_parameters(model)

print(f"{'component':<28}{'parameters':>14}{'share':>9}")
print("-" * 51)
for name, n in (("encoder (conv stages)", enc), ("classifier head", head)):
    print(f"{name:<28}{n:>14,}{n/total:>9.1%}")
print("-" * 51)
print(f"{'total':<28}{total:>14,}")
print(f"{'trainable':<28}{trainable:>14,}")

flat = 16 * 16 * 256 * 128
print(f"\nif the head flattened instead of pooling, its first layer alone would be")
print(f"{flat:,} weights -- {flat/total:.0f}x the entire network as built.")

component                       parameters    share
---------------------------------------------------
encoder (conv stages)            1,126,368    96.5%
classifier head                     41,347     3.5%
---------------------------------------------------
total                            1,167,715
trainable                        1,167,715

if the head flattened instead of pooling, its first layer alone would be
8,388,608 weights -- 7x the entire network as built.


In [ ]:
# 3. RECEPTIVE FIELD — THE CAPACITY LIMIT THAT IS NOT PARAMETER COUNT
"""
Adding the second convolution at stages 3 and 4 adds parameters, but that is not
the argument for it. The argument is receptive field: how much of the input a
single unit at the deepest layer can see.

A 3x3 convolution adds 2*jump to the receptive field; a 2x2 stride-2 pool adds
1*jump and then doubles the jump. Walking the stack that way gives 38px for the
plain network and 62px for the deeper one, on a 128px input.

38px is a quarter of the image. A unit seeing that much cannot judge a lesion's
margin against surrounding tissue, or its position relative to the midline, and
both of those are what distinguishes the tumour types from one another. 62px is
roughly half the scan, which is enough for that kind of comparison.

"""
print(f"{'variant':<20}{'params':>12}{'receptive field':>18}{'share of 128px':>17}")
print("-" * 67)
for deep in (False, True):
    m = BrainTumourNet(len(CLASSES), dropout=DROPOUT, deep=deep)
    rf = receptive_field(deep)
    print(f"{'deep=' + str(deep):<20}{count_parameters(m)[0]:>12,}"
          f"{rf:>15}px{rf/IMG_SIZE:>17.0%}")

rf_growth, jump, rfs = 1, 1, []
for layer in ["conv", "pool", "conv", "pool", "conv", "conv", "pool", "conv", "conv"]:
    if layer == "conv":
        rf_growth += 2 * jump
    else:
        rf_growth += jump; jump *= 2
    rfs.append((layer, rf_growth))

fig, ax = viz.styled_fig(figsize=(7.5, 3.8))
ax.step(range(1, len(rfs) + 1), [r for _, r in rfs], where='mid',
        color=config.PALETTE["train"], lw=2)
ax.axhline(IMG_SIZE, color='k', ls='--', lw=1, alpha=0.6, label=f"input {IMG_SIZE}px")
ax.axhline(receptive_field(False), color=config.PALETTE["val"], ls=':', lw=1.5,
           label=f"plain stack ends at {receptive_field(False)}px")
ax.set_xticks(range(1, len(rfs) + 1))
ax.set_xticklabels([l for l, _ in rfs], rotation=45, ha='right', fontsize=8)
ax.set_ylabel("receptive field (px)"); ax.set_facecolor(config.FACE)
ax.set_title("Receptive field grows through the stack", fontsize=11, fontweight='bold')
ax.legend(fontsize=8)
plt.tight_layout(); viz.save(fig, "receptive_field.png")

variant                   params   receptive field   share of 128px
-------------------------------------------------------------------
deep=False               429,667             38px              30%
deep=True              1,167,715             62px              48%
  saved -> outputs/receptive_field.png


WindowsPath('C:/Games/Codes/Python/Projects/Brain_Tumour_Detection/Final_Project/outputs/receptive_field.png')

In [ ]:
# 4. THE SHAPES, READ RATHER THAN ASSUMED
"""
A network can be wrong in a way that still runs. Hooks report what each stage
actually produces, which is cheaper than reasoning about it and does not go
stale when the architecture changes.

"""
shapes = {}
for name in ("block1", "block2", "block3", "block3b", "block4", "block4b"):
    mod = getattr(model.features, name, None)
    if mod is not None:
        mod.register_forward_hook(
            lambda m, i, o, k=name: shapes.__setitem__(k, tuple(o.shape)[1:]))

model.eval()
with torch.no_grad():
    logits = model(torch.zeros(2, 1, IMG_SIZE, IMG_SIZE))

print(f"input                 (1, {IMG_SIZE}, {IMG_SIZE})")
for name, shape in shapes.items():
    print(f"  after {name:<12}{shape}")
print(f"  after gap           (256,)")
print(f"  logits              ({logits.shape[1]},)  one per class\n")
print(f"deepest feature map is {shapes['block4b'][1]}x{shapes['block4b'][2]} "
      f"-- the resolution any CAM taken there will have")

input                 (1, 128, 128)
  after block1      (32, 128, 128)
  after block2      (64, 64, 64)
  after block3      (128, 32, 32)
  after block3b     (128, 32, 32)
  after block4      (256, 16, 16)
  after block4b     (256, 16, 16)
  after gap           (256,)
  logits              (3,)  one per class

deepest feature map is 16x16 -- the resolution any CAM taken there will have


In [ ]:
# 5. A FORWARD AND BACKWARD PASS
model.train()
x = torch.randn(8, 1, IMG_SIZE, IMG_SIZE)
y = torch.randint(0, len(CLASSES), (8,))
loss = torch.nn.functional.cross_entropy(model(x), y)
model.zero_grad(); loss.backward()

first = model.features.block1.conv.weight
last  = model.head.classifier.weight
print(f"loss on random input          {loss.item():.4f}   "
      f"(ln({len(CLASSES)}) = {np.log(len(CLASSES)):.4f} expected at init)")
print(f"gradient at first convolution {first.grad.abs().mean():.3e}")
print(f"gradient at final linear      {last.grad.abs().mean():.3e}")
print("\n-> gradient reaches the first layer, so the graph is connected")

loss on random input          1.0659   (ln(3) = 1.0986 expected at init)
gradient at first convolution 6.320e-04
gradient at final linear      1.206e-02

-> gradient reaches the first layer, so the graph is connected


In [7]:
# 6. VERIFICATION
"""
The checks that fail loudly and name what failed. The last one is the one that
matters for the report: two freshly built models must differ, because identical
weights across constructions would mean something was loaded rather than
initialised.
"""
plain = BrainTumourNet(len(CLASSES), dropout=DROPOUT, deep=False)
deep  = BrainTumourNet(len(CLASSES), dropout=DROPOUT, deep=True)

torch.manual_seed(1); a = BrainTumourNet(len(CLASSES), deep=True)
torch.manual_seed(2); b = BrainTumourNet(len(CLASSES), deep=True)
identical = all(torch.equal(p, q) for p, q in
                zip(a.features.parameters(), b.features.parameters()))

elu = BrainTumourNet(len(CLASSES), deep=True, activation="elu")

checks = [
    ("feature maps halve at each pool",
     shapes["block1"][1] == IMG_SIZE and shapes["block4"][1] == IMG_SIZE // 8),
    ("head is a small fraction of parameters", head / total < 0.10),
    ("deeper variant raises receptive field",
     receptive_field(True) > receptive_field(False)),
    ("receptive field computed, not hardcoded",
     (receptive_field(False), receptive_field(True)) == (38, 62)),
    ("logits are one per class", logits.shape[1] == len(CLASSES)),
    ("gradient reaches the first convolution", first.grad.abs().mean() > 0),
    ("ELU variant builds with matched init",
     count_parameters(elu)[0] == count_parameters(deep)[0]),
    ("no pretrained weights: two builds differ", not identical),
]
print("=" * 62)
print("PHASE 2 VERIFICATION")
print("=" * 62)
for label, ok in checks:
    print(f"  {'OK  ' if ok else 'FAIL'}  {label}")
failed = [label for label, ok in checks if not ok]
assert not failed, "failed checks: " + "; ".join(failed)

print(f"""
  architecture   1 -> 32 -> 64 -> 128 -> 256 -> GAP -> 128 -> 64 -> {len(CLASSES)}
  parameters     {count_parameters(plain)[0]:,} plain / {count_parameters(deep)[0]:,} deep
  receptive fld  {receptive_field(False)}px plain / {receptive_field(True)}px deep, on a {IMG_SIZE}px input
  activation     relu (elu available, to be measured in Phase 4)
  device         {DEVICE}

  PREDICTION, recorded before Phase 4 measures it: the deeper variant wins, and
  it wins because 38px is too little context to judge a lesion's margin -- not
  because it has more parameters.

  Nothing is trained here. Phase 3 takes this model and the Phase 1 split.""")

PHASE 2 VERIFICATION
  OK    feature maps halve at each pool
  OK    head is a small fraction of parameters
  OK    deeper variant raises receptive field
  OK    receptive field computed, not hardcoded
  OK    logits are one per class
  OK    gradient reaches the first convolution
  OK    ELU variant builds with matched init
  OK    no pretrained weights: two builds differ

  architecture   1 -> 32 -> 64 -> 128 -> 256 -> GAP -> 128 -> 64 -> 3
  parameters     429,667 plain / 1,167,715 deep
  receptive fld  38px plain / 62px deep, on a 128px input
  activation     relu (elu available, to be measured in Phase 4)
  device         cuda

  PREDICTION, recorded before Phase 4 measures it: the deeper variant wins, and
  it wins because 38px is too little context to judge a lesion's margin -- not
  because it has more parameters.

  Nothing is trained here. Phase 3 takes this model and the Phase 1 split.
